In [ ]:
use std::collections::HashMap;
type Id = usize;
type PId = usize;
type Subst = Vec<Option<Id>>;
#[derive(Debug, Clone, PartialEq, Eq, Hash)]
struct Node {
    f : String,
    args : Vec<Id>,
}

#[derive(Debug, Default)]
struct EGraph {
    memo : HashMap<Node, Id>,
    nodes : Vec<Node>,
    uf : Vec<Id>
    // bottom up patterns is kind of an analysis
    patterns : Vec<Vec<(PId, Subst)>> // which subpatterns this eid can match. 
    patmemo : HashMap<Pattern, PId>
    pnodes : Vec<Pattern>

}

impl EGraph {
    fn app(&mut self, f : &str, args : Vec<Id>) -> Id {
        let node = Node { f: f.to_string(), args };
        if let Some(&id) = self.memo.get(&node) {
            id
        } else {
            let id = self.nodes.len();
            self.nodes.push(node.clone());
            self.memo.insert(node, id);
            self.uf.push(id);
            id
        }
    }

    fn find(&mut self, mut id : Id) -> Id {
        while self.uf[id] != id {
            id = self.uf[id];
        }
        id
    }

    fn union(&mut self, a : Id, b : Id) {
        let ra = self.find(a);
        let rb = self.find(b);
        if ra != rb {
            self.uf[ra] = rb;
        }
    }
}

let mut hc = EGraph::default();


assert!(3 == 3);

let a = hc.app("a", vec![]);
let b = hc.app("b", vec![]);
let fa = hc.app("f", vec![a]);
let a1 = hc.app("a", vec![]);
assert!(a == a1);




eager bottom up ematching



In [ ]:
type VId = String; // &'static str  might be nice for prototyping
#[derive(Debug, Clone, PartialEq, Eq, Hash)]
enum Term {
    Var(VId),
    App(String, Vec<Term>),
}
fn var(v: impl Into<String>) -> Term {Term::Var(v.into())}
fn app(f: impl Into<String>, args: Vec<Term>) -> Term {Term::App(f.into(), args)}
fn s(name : impl Into<String>) -> String {name.into()}
// use Term::*;
type Subst = HashMap<VId, Term>;
fn subst(t : &Term, s : &Subst) -> Term {
    match t {
        Term::Var(v) => {
            if let Some(t) = s.get(v) {
                subst(t, s)
            } else {
                Term::Var(v.into())
            }
        }
        Term::App(f, args) => {
            Term::App(f.clone(), args.iter().map(|t| subst(t, s)).collect())
        }
    }
}
let x = var("x");
let y = var("y");
let a = app("a", vec![]);
let fxy = app("f", vec![x.clone(), y.clone()]);
let subst : Subst = HashMap::from([
    (s("x"), a.clone())
]);

In [ ]:

//subst.insert("x".to_string(), Term::app("a", vec![]));

fn unify(t1 : Term, t2 : Term) -> Option<Subst> {
    let mut todo = vec![(t1, t2)];
    let mut subst = HashMap::new()
    while let Some((t1, t2)) = todo.pop() {
        // substitute into t1 and t2
        let t1 = subst(&t1, &subst);
        let t2 = subst(&t2, &subst);
        match (t1, t2) {
            (Term::Var(v), t) | (t, Term::Var(v)) => {
                if let Some(t3) = subst.get(v) {
                    todo.push((t3, t));
                } else {
                    subst.insert(*v, t.clone());
                }

            }
        }
}
}

/*
fn unify()


struct PrologState {

}
*/




# trying z3 / cvc5

Yes, it's ok ish to load in z3. 7s loading time. But Maybe pure rust std stuff is btter.I do lot's of that sort of thing too.



In [3]:
:dep verus

Error: Dependency `verus` is missing a lib target

In [ ]:
use vstd::prelude::*;

verus! {

spec fn min(x: int, y: int) -> int {
    if x <= y {
        x
    } else {
        y
    }
}

fn main() {
    assert(min(10, 20) == 10);
    assert(min(-10, -20) == -20);
    assert(forall|i: int, j: int| min(i, j) <= i && min(i, j) <= j);
    assert(forall|i: int, j: int| min(i, j) == i || min(i, j) == j);
    assert(forall|i: int, j: int| min(i, j) == min(j, i));
}

} // verus!

In [2]:
:dep petgraph

In [2]:
:dep smtlib

In [3]:
use smtlib::{backend::z3_binary::Z3Binary, Int, SatResultWithModel, Solver, Storage, prelude::*};

{
let st = Storage::new();

// Initialize the solver with the Z3 backend. The "z3" string refers the
// to location of the already installed `z3` binary. In this case, the
// binary is in the path.
let mut solver = Solver::new(&st, Z3Binary::new("z3")?)?;

// Declare two new variables
let x = Int::new_const(&st, "x");
let y = Int::new_const(&st, "y");

// Assert some constraints. This tells the solver that these expressions
// must be true, so any solution will satisfy these.
solver.assert(x._eq(y + 25))?;
solver.assert(x._eq(204))?;
}

()

In [ ]:
// :sccache 1  deprecated I think

Error: Couldn't find sccache. Try running `cargo install sccache`.

In [2]:
:cache 500

cache: 500 MiB


In [2]:
println!("Hello, world!");

Hello, world!


https://github.com/cvc5/cvc5-rs

In [2]:
:dep cvc5-rs

In [6]:
use cvc5_rs::{TermManager, Solver, Kind};

let tm = TermManager::new();
let mut solver = Solver::new( & tm);

solver.set_logic("QF_LIA");
solver.set_option("produce-models", "true");

let int_sort = tm.integer_sort();
let x = tm.mk_const(int_sort, "x");
let zero = tm.mk_integer(0);

let gt = tm.mk_term(Kind::CVC5_KIND_GT, & [x.clone(), zero]);
solver.assert_formula(gt);

let result = solver.check_sat();
assert!(result.is_sat());

let x_val = solver.get_value(x);
println!("x = {x_val}");

x = 1


In [ ]:
:dep z3 = { version = "0.19.5", default-features = false }
// Oof so I have to rebuild every time? That's somewhat painful. I added :cache 1000 to init file and that improved 2x maybe
// Yea it's not better


In [3]:
use z3::ast::{Int, Bool};
let p = Bool::new_const("p");
&p & &p;
p.eq(&p)
// A macro for building expr might be nice
// z3!(p & p)

(= p p)

In [4]:
use z3::{Config, Context, Solver};

In [23]:
:help

:allow_static_linking,Set whether to allow static linking of dependencies (0/1)
:build_env,Set environment variables when building code (key=value)
:cache,"Set cache size in MiB, or 0 to disable."
:clear,"Clear all state, keeping compilation cache"
:clear_cache,Clear the cache used by the :cache command
:codegen_backend,Set/print the codegen backend. Requires nightly
:dep,"Add dependency. e.g. :dep regex = ""1.0"""
:doc,"show the documentation of a variable, keyword, type or module"
:efmt,Set the formatter for errors returned by ?
:env,Set an environment variable (key=value)
:explain,Print explanation of last error


(= p p)

In [ ]:
let mut s = Solver::new();
s.